# Fine-tune bge-small on CAR-T training triplets (free Colab GPU)

Same settings as `train/finetune.py`, just on a GPU.

**Before running:** menu **Runtime → Change runtime type → T4 GPU → Save**.

Then run each cell in order (▶ or Shift+Enter). It takes about 5 minutes in total.

In [ ]:
# 1. Install the same sentence-transformers version as the project
!pip -q install sentence-transformers==6.1.0 datasets accelerate

In [ ]:
# 2. Check the GPU is on (should print True and a GPU name)
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 3. Upload train/train_split.jsonl from the project folder
from google.colab import files
uploaded = files.upload()
print(list(uploaded))

In [ ]:
# 4. Train (about 2-4 minutes on a T4)
import json
from datasets import Dataset
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.sentence_transformer import losses
from sentence_transformers.training_args import BatchSamplers

QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '
rows = [json.loads(l) for l in open('train_split.jsonl')]
print(len(rows), 'training rows')
dataset = Dataset.from_dict({
    'anchor': [QUERY_PREFIX + r['question'] for r in rows],
    'positive': [r['positive'] for r in rows],
    'negative': [r['negative'] for r in rows],
})
model = SentenceTransformer('BAAI/bge-small-en-v1.5', device='cuda')
args = SentenceTransformerTrainingArguments(
    output_dir='checkpoints', num_train_epochs=3, per_device_train_batch_size=32, learning_rate=2e-5,
    warmup_ratio=0.1, batch_sampler=BatchSamplers.NO_DUPLICATES, save_strategy='no', logging_steps=5,
    report_to='none', seed=42, fp16=True,
)
loss = losses.CachedMultipleNegativesRankingLoss(model, mini_batch_size=16)
SentenceTransformerTrainer(model=model, args=args, train_dataset=dataset, loss=loss).train()
model.save('bge-small-cart')
print('saved')

In [ ]:
# 5. Download the trained model (a zip file of about 120 MB)
!zip -q -r bge-small-cart.zip bge-small-cart
files.download('bge-small-cart.zip')

**After downloading:** move `bge-small-cart.zip` into the project's `train/models/` folder and tell Claude. It will unzip it, check the results, and rebuild the search with the new model.